# 03. Tuning & Run (Optuna + N_REPEATS 최종 학습)

01에서 만든 FuxiCTR parquet을 사용해 `MODEL`을 튜닝하고, best params로 N_REPEATS번 재학습 뒤
메트릭과 test 예측을 저장한다.

흐름:
1. `DATASET` / `MODEL` / `N_TRIALS` / `N_REPEATS` 스위치만 설정
2. `configs/models/{MODEL}_base.yaml` + `configs/tuning/{MODEL}_search.yaml` 로드
3. Optuna study (SQLite `artifacts/tuning/...` 저장 → 중단/재개 가능)
4. best params로 N_REPEATS번 재학습 → 메트릭 + test 예측 저장
5. 저장된 run 요약 출력

산출물:
- `artifacts/runs/{dataset_id}/{model_id}.model` (FuxiCTR best checkpoint)
- `artifacts/runs/{dataset_id}/{model_id}.metrics.json` / `.params.json`
- `artifacts/predictions/{dataset_id}/{MODEL}/{run_id}/preds.parquet`
- `artifacts/tuning/{dataset_id}/{MODEL}/study.db` (Optuna, 재개 가능)

In [1]:
import sys, json, copy
from pathlib import Path
import pandas as pd

_HERE = Path.cwd()
if str(_HERE) not in sys.path:
    sys.path.insert(0, str(_HERE))
from utils import nb_utils as U
U.add_fuxictr_to_path()

import optuna
print('optuna =', optuna.__version__)

optuna = 4.8.0


## 1. 스위치

다른 실험을 돌리려면 이 셀만 바꾸면 된다.

In [2]:
DATASET_ID = 'TAAC2026'   # 01에서 빌드된 dataset_id
MODEL      = 'DIN'                     # configs/models/{MODEL}_base.yaml 에 정의된 이름
N_TRIALS   = 3                        # Optuna trial 수 (smoke test는 3~5)
N_REPEATS  = 2                         # best params로 재학습할 횟수 (seed만 다름)
SEEDS      = [2026, 2027, 2028, 2029, 2030][:N_REPEATS]
TUNE_EPOCHS = 3                        # 튜닝 trial당 epoch (빨리 탐색)

# study DB (SQLite) 위치 — 존재하면 자동 resume
study_dir = U.ARTIFACT_ROOT / 'tuning' / DATASET_ID / MODEL
study_dir.mkdir(parents=True, exist_ok=True)
STUDY_DB = study_dir / 'study.db'
STUDY_NAME = f'{MODEL}__{DATASET_ID}'
print(f'dataset={DATASET_ID}  model={MODEL}  trials={N_TRIALS}  repeats={N_REPEATS}')
print(f'study  = sqlite:///{STUDY_DB}')

dataset=TAAC2026  model=DIN  trials=3  repeats=2
study  = sqlite:////NAS/hyunwoong/Hayanmind-project/notebook/artifacts/tuning/TAAC2026/DIN/study.db


## 2. Base config + search space 로드

- `load_base_config(MODEL, DATASET_ID)` → `{MODEL}_base.yaml`의 defaults → Base → Base@{DATASET_ID} → MODEL → MODEL@{DATASET_ID} 순으로 머지
  (데이터셋별 `feature_specs` / `*_target_field` / `*_sequence_field` 오버라이드가 자동 적용됨)
- `load_search_space(MODEL)` → `{MODEL}_search.yaml` 의 탐색 공간

In [3]:
base_cfg = U.load_base_config(MODEL, DATASET_ID)
search_space = U.load_search_space(MODEL)

# feature_map이 있어야 data 경로 검증이 가능
fmap = U.load_feature_map(DATASET_ID)
print(f'base_cfg keys (일부): {sorted(base_cfg)[:12]} ...')
print(f'search_space       : {list(search_space)}')
print(f'feature_map labels : {fmap["labels"]}  total_features={fmap["total_features"]}')

base_cfg keys (일부): ['attention_dropout', 'attention_hidden_activations', 'attention_hidden_units', 'attention_output_activation', 'batch_norm', 'batch_size', 'debug_mode', 'din_sequence_field', 'din_target_field', 'din_use_softmax', 'dnn_activations', 'dnn_hidden_units'] ...
search_space       : ['learning_rate', 'embedding_dim', 'dnn_hidden_units', 'net_dropout', 'attention_dropout', 'batch_size']
feature_map labels : ['label']  total_features=1530322


## 3. Optuna objective

- 각 trial마다 `{MODEL}_tune_t{trial_number}` 로 `run_training` 호출 (test 예측 저장 X)
- valid AUC를 반환 → Optuna가 maximize
- 실패한 trial은 `optuna.TrialPruned` 로 스킵

⚠️ 튜닝 중에도 checkpoint는 `artifacts/runs/{dataset_id}/` 아래에 쌓이므로,
나중에 정리하려면 `{MODEL}_tune_*` 패턴으로 삭제하면 된다.

In [4]:
def objective(trial: 'optuna.Trial') -> float:
    suggested = U.suggest_from_space(trial, search_space)
    params = copy.deepcopy(base_cfg)
    params.update(suggested)
    params['epochs'] = TUNE_EPOCHS
    params['save_best_only'] = False  # 튜닝 중 checkpoint IO 최소화
    run_id = f'tune_t{trial.number:03d}_s{params.get("seed", 2026)}'
    try:
        summary = U.run_training(
            dataset_id=DATASET_ID,
            model_name=MODEL,
            params=params,
            run_id=run_id,
            save_test_preds=False,
        )
    except Exception as e:
        print(f'[trial {trial.number}] failed: {e!r}')
        raise optuna.TrialPruned() from e
    auc = summary.get('valid_AUC')
    if auc is None:
        raise optuna.TrialPruned()
    print(f'[trial {trial.number}] valid_AUC={auc:.5f}  params={suggested}')
    return float(auc)

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=f'sqlite:///{STUDY_DB}',
    direction='maximize',
    load_if_exists=True,
)
print(f'study loaded. prior trials = {len(study.trials)}')

[I 2026-04-19 15:29:11,675] A new study created in RDB with name: DIN__TAAC2026


study loaded. prior trials = 0


In [5]:
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
print('\n=== Optuna summary ===')
print('best value  =', study.best_value)
print('best params =', study.best_params)
df_trials = study.trials_dataframe(attrs=('number','value','state','params','datetime_complete'))
df_trials.tail(min(20, len(df_trials)))

2026-04-19 15:29:13,573 P2399999 INFO FuxiCTR version: 2.3.9
2026-04-19 15:29:13,575 P2399999 INFO Run tune_t000_s2026: {
    "attention_dropout": "0.2150721079075201",
    "attention_hidden_activations": "Dice",
    "attention_hidden_units": "[64]",
    "attention_output_activation": "None",
    "batch_norm": "False",
    "batch_size": "4096",
    "data_format": "parquet",
    "data_root": "/NAS/hyunwoong/Hayanmind-project/data",
    "dataset_id": "TAAC2026",
    "debug_mode": "False",
    "din_sequence_field": "item_int_feats_11",
    "din_target_field": "item_id",
    "din_use_softmax": "False",
    "dnn_activations": "relu",
    "dnn_hidden_units": "[512, 256, 128]",
    "early_stop_patience": "2",
    "embedding_dim": "32",
    "embedding_regularizer": "0",
    "epochs": "3",
    "eval_steps": "None",
    "feature_config": "None",
    "feature_specs": "[{'name': 'item_int_feats_11', 'feature_encoder': None}]",
    "gpu": "0",
    "group_id": "None",
    "learning_rate": "0.0007604

  0%|          | 0/1 [00:00<?, ?it/s]

2026-04-19 15:29:19,297 P2399999 INFO Train loss: 13.809524
2026-04-19 15:29:19,300 P2399999 INFO Evaluation @epoch 1 - batch 1: 


100%|██████████| 1/1 [00:00<00:00,  4.16it/s]

2026-04-19 15:29:19,563 P2399999 INFO [Metrics] AUC: 0.500000


...

2026-04-19 15:29:56,605 P2399999 INFO [Metrics] AUC: 0.500000 - logloss: 14.130197



[run tune_t002_s2026] valid=OrderedDict([('AUC', 0.5), ('logloss', 13.815510572701148)])  test=OrderedDict([('AUC', 0.5), ('logloss', 14.130197200134901)])
[run tune_t002_s2026] artifacts -> /NAS/hyunwoong/Hayanmind-project/notebook/artifacts/runs/TAAC2026/DIN_tune_t002_s2026.*
[trial 2] valid_AUC=0.50000  params={'learning_rate': 0.0009891936374539197, 'embedding_dim': 32, 'dnn_hidden_units': [1024, 512, 256], 'net_dropout': 0.26968945788027004, 'attention_dropout': 0.21761552589013844, 'batch_size': 8192}


[I 2026-04-19 15:29:56,674] Trial 2 finished with value: 0.5 and parameters: {'learning_rate': 0.0009891936374539197, 'embedding_dim': 32, 'dnn_hidden_units': '[1024, 512, 256]', 'net_dropout': 0.26968945788027004, 'attention_dropout': 0.21761552589013844, 'batch_size': 8192}. Best is trial 0 with value: 0.5.



=== Optuna summary ===
best value  = 0.5
best params = {'learning_rate': 0.0007604734065844932, 'embedding_dim': 32, 'dnn_hidden_units': '[512, 256, 128]', 'net_dropout': 0.04846728945740564, 'attention_dropout': 0.2150721079075201, 'batch_size': 4096}


,number,value,state,params_attention_dropout,params_batch_size,params_dnn_hidden_units,params_embedding_dim,params_learning_rate,params_net_dropout,datetime_complete
0,0,0.5,COMPLETE,0.215072,4096,"[512, 256, 128]",32,0.000760,0.048467,2026-04-19 15:29:29.056409
1,1,0.5,COMPLETE,0.253341,4096,"[512, 256, 128]",32,0.000780,0.004588,2026-04-19 15:29:42.662278
2,2,0.5,COMPLETE,0.217616,8192,"[1024, 512, 256]",32,0.000989,0.269689,2026-04-19 15:29:56.654861


## 4. best params로 최종 학습 (N_REPEATS)

seed만 다르게 `N_REPEATS`번 재학습한다. 이때는 `save_test_preds=True` 로
test 예측을 parquet에 저장 → 04 분석 노트북에서 바로 로드 가능.

In [6]:
final_summaries = []
for seed in SEEDS:
    params = copy.deepcopy(base_cfg)
    params.update(U.decode_best_params(study.best_params))
    params['seed'] = seed
    run_id = U.make_run_id(MODEL, DATASET_ID, seed)
    print(f'\n=== FINAL RUN seed={seed}  run_id={run_id} ===')
    summary = U.run_training(
        dataset_id=DATASET_ID,
        model_name=MODEL,
        params=params,
        run_id=run_id,
        save_test_preds=True,
    )
    final_summaries.append(summary)

final_df = pd.DataFrame(final_summaries)
print('\n=== final runs ===')
final_df


=== FINAL RUN seed=2026  run_id=20260419-152956_DIN_TAAC2026_s2026 ===


2026-04-19 15:29:56,802 P2399999 INFO FuxiCTR version: 2.3.9
2026-04-19 15:29:56,804 P2399999 INFO Run 20260419-152956_DIN_TAAC2026_s2026: {
    "attention_dropout": "0.2150721079075201",
    "attention_hidden_activations": "Dice",
    "attention_hidden_units": "[64]",
    "attention_output_activation": "None",
    "batch_norm": "False",
    "batch_size": "4096",
    "data_format": "parquet",
    "data_root": "/NAS/hyunwoong/Hayanmind-project/data",
    "dataset_id": "TAAC2026",
    "debug_mode": "False",
    "din_sequence_field": "item_int_feats_11",
    "din_target_field": "item_id",
    "din_use_softmax": "False",
    "dnn_activations": "relu",
    "dnn_hidden_units": "[512, 256, 128]",
    "early_stop_patience": "2",
    "embedding_dim": "32",
    "embedding_regularizer": "0",
    "epochs": "20",
    "eval_steps": "None",
    "feature_config": "None",
    "feature_specs": "[{'name': 'item_int_feats_11', 'feature_encoder': None}]",
    "gpu": "0",
    "group_id": "None",
    "learni

  0%|          | 0/1 [00:00<?, ?it/s]

2026-04-19 15:30:00,323 P2399999 INFO Train loss: 13.809524
2026-04-19 15:30:00,326 P2399999 INFO Evaluation @epoch 1 - batch 1: 


100%|██████████| 1/1 [00:00<00:00,  6.66it/s]

...

2026-04-19 15:30:15,840 P2399999 INFO [Metrics] AUC: 0.500000 - logloss: 2.302585
2026-04-19 15:30:15,844 P2399999 INFO *** Test evaluation (run=20260419-153006_DIN_TAAC2026_s2027) ***
2026-04-19 15:30:15,846 P2399999 INFO Loading datasets...
2026-04-19 15:30:15,929 P2399999 INFO Test samples: total/300, blocks/1
2026-04-19 15:30:15,930 P2399999 INFO Loading test data done.



100%|██████████| 1/1 [00:00<00:00,  3.74it/s]

2026-04-19 15:30:16,220 P2399999 INFO [Metrics] AUC: 0.500000 - logloss: 1.987899
2026-04-19 15:30:16,227 P2399999 INFO Loading datasets...
2026-04-19 15:30:16,303 P2399999 INFO Test samples: total/300, blocks/1
2026-04-19 15:30:16,304 P2399999 INFO Loading test data done.



100%|██████████| 1/1 [00:00<00:00,  3.66it/s]
[preds] saved 300 rows -> /NAS/hyunwoong/Hayanmind-project/notebook/artifacts/predictions/TAAC2026/DIN/20260419-153006_DIN_TAAC2026_s2027/preds.parquet
[run 20260419-153006_DIN_TAAC2026_s2027] valid=OrderedDict([('AUC', 0.5), ('logloss', 2.302585178708335)])  test=OrderedDict([('AUC', 0.5), ('logloss', 1.9878985512848644)])
[run 20260419-153006_DIN_TAAC2026_s2027] artifacts -> /NAS/hyunwoong/Hayanmind-project/notebook/artifacts/runs/TAAC2026/DIN_20260419-153006_DIN_TAAC2026_s2027.*
[run 20260419-153006_DIN_TAAC2026_s2027] preds    -> /NAS/hyunwoong/Hayanmind-project/notebook/artifacts/predictions/TAAC2026/DIN/20260419-153006_DIN_TAAC2026_s2027/preds.parquet

=== final runs ===


,model,dataset_id,run_id,model_id,train_seconds,valid_AUC,valid_logloss,test_AUC,test_logloss
0,DIN,TAAC2026,20260419-152956_DIN_TAAC2026_s2026,DIN_20260419-152956_DIN_TAAC2026_s2026,5.38,0.5,2.302585,0.5,1.987899
1,DIN,TAAC2026,20260419-153006_DIN_TAAC2026_s2027,DIN_20260419-153006_DIN_TAAC2026_s2027,5.68,0.5,2.302585,0.5,1.987899


## 5. 저장된 run 요약 출력 (list_runs)

`artifacts/runs/{DATASET_ID}/` 아래에 쌓인 모든 run(튜닝 trial 포함)을 메트릭 표로.

In [7]:
runs = U.list_runs(DATASET_ID)
cols = [c for c in ['dataset','model','run_id','metric.valid_AUC','metric.test_AUC','metric.valid_logloss','metric.test_logloss','metric.train_seconds'] if c in runs.columns]
runs[cols].sort_values(cols[-2] if 'metric.test_AUC' in cols else 'run_id', ascending=False).head(30)

,dataset,model,run_id,metric.valid_AUC,metric.test_AUC,metric.valid_logloss,metric.test_logloss,metric.train_seconds
4,TAAC2026,DIN,tune_t002_s2026,0.5,0.5,13.815511,14.130197,10.06
0,TAAC2026,DIN,20260419-152956_DIN_TAAC2026_s2026,0.5,0.5,2.302585,1.987899,5.38
1,TAAC2026,DIN,20260419-153006_DIN_TAAC2026_s2027,0.5,0.5,2.302585,1.987899,5.68
2,TAAC2026,DIN,tune_t000_s2026,0.5,0.5,2.302585,1.987899,9.79
3,TAAC2026,DIN,tune_t001_s2026,0.5,0.5,2.302585,1.987899,9.53
